In [1]:
# imports
import json
import re
from collections import Counter

In [2]:
# read in the raw recipes
path_raw = "../recipe_crawler/recipes.json"
with open(path_raw, 'r') as f:
    recipes = json.load(f)

# print the first few recipes
print(json.dumps(recipes[0:2], indent=4))

[
    {
        "recipe_name": "Nagi\u2019s Salad Sandwich",
        "ingredient_groups": [
            {
                "group_name": null,
                "ingredients": [
                    {
                        "amount": "1",
                        "unit": "",
                        "name": "soft bread roll",
                        "notes": ", wholemeal or white (or sliced bread, wraps)"
                    },
                    {
                        "amount": "1",
                        "unit": "",
                        "name": "small carrot",
                        "notes": ", peeled, grated using a box grater (3/4 cup lightly packed per sandwich)"
                    },
                    {
                        "amount": "1/2",
                        "unit": "",
                        "name": "ripe avocado",
                        "notes": ""
                    },
                    {
                        "amount": "1",
                        "unit

In [3]:
with open("../recipe_crawler/recipes.json", "r") as f:
    recipes = json.load(f)

# Count ingredient groups per recipe
group_counts = Counter()
for r in recipes:
    num_groups = len(r["ingredient_groups"])
    group_counts[num_groups] += 1

print("Distribution of ingredient groups per recipe:\n")
print(f"{'# Groups':<10} {'# Recipes':<12} {'%'}")
print("-" * 30)
for num_groups in sorted(group_counts):
    count = group_counts[num_groups]
    pct = (count / len(recipes)) * 100
    bar = "█" * int(pct / 2)
    print(f"{num_groups:<10} {count:<12} {pct:.1f}%  {bar}")

print(f"\nTotal recipes: {len(recipes)}")
print(f"Mean groups per recipe: {sum(k * v for k, v in group_counts.items()) / len(recipes):.1f}")
print(f"Range: {min(group_counts)} – {max(group_counts)}")

Distribution of ingredient groups per recipe:

# Groups   # Recipes    %
------------------------------
1          286          19.3%  █████████
2          486          32.8%  ████████████████
3          415          28.0%  ██████████████
4          207          14.0%  ██████
5          59           4.0%  █
6          18           1.2%  
7          6            0.4%  
8          1            0.1%  
9          2            0.1%  
10         1            0.1%  
13         1            0.1%  

Total recipes: 1482
Mean groups per recipe: 2.6
Range: 1 – 13


In [4]:
import json

with open("../recipe_crawler/recipes.json", "r") as f:
    recipes = json.load(f)

# Counters
only_group_null = 0      # Recipe has 1 group total, and it's None
one_of_many_null = 0     # Recipe has multiple groups, exactly 1 is None
multiple_nulls = 0       # Recipe has multiple groups, 2+ are None

for r in recipes:
    groups = r["ingredient_groups"]
    total = len(groups)
    null_count = sum(1 for g in groups if g["group_name"] is None)
    
    if null_count == 0:
        continue
    elif total == 1 and null_count == 1:
        only_group_null += 1
    elif total > 1 and null_count == 1:
        one_of_many_null += 1
    else:
        multiple_nulls += 1

print("Ingredient groups with null names:\n")
print(f"  Only group in recipe, and it's null:  {only_group_null}")
print(f"  One null among several named groups:   {one_of_many_null}")
print(f"  Multiple nulls in same recipe:         {multiple_nulls}")
print(f"  ─────────────────────────────────────")
print(f"  Total recipes with null groups:        {only_group_null + one_of_many_null + multiple_nulls}")
print(f"  Total recipes overall:                 {len(recipes)}")

Ingredient groups with null names:

  Only group in recipe, and it's null:  282
  One null among several named groups:   605
  Multiple nulls in same recipe:         8
  ─────────────────────────────────────
  Total recipes with null groups:        895
  Total recipes overall:                 1482


In [5]:
count = 0
for r in recipes:
    groups = r["ingredient_groups"]
    null_positions = [i for i, g in enumerate(groups) if g["group_name"] is None]
    
    if len(null_positions) >= 2:
        count += 1
        print("=" * 70)
        print(f"RECIPE {count}: {r['recipe_name']}")
        print(f"Total groups: {len(groups)},  Null groups: {len(null_positions)}")
        print(f"Null at positions: {null_positions}")
        print("=" * 70)
        
        for i, g in enumerate(groups):
            marker = " ← NULL" if g["group_name"] is None else ""
            print(f"\n  [{i}] group_name: {g['group_name']}{marker}")
            print(f"      group_type: {g.get('group_type', 'N/A')}")
            print(f"      ingredients ({len(g['ingredients'])}):")
            for ing in g["ingredients"]:
                print(f"        {ing['amount']} {ing['unit']} {ing['name']}".strip())
        
        print()

print(f"\nTotal: {count} recipes with multiple null groups")

RECIPE 1: One pot chicken risoni with crispy salami (orzo)
Total groups: 4,  Null groups: 2
Null at positions: [0, 3]

  [0] group_name: None ← NULL
      group_type: N/A
      ingredients (3):
1/2 tbsp olive oil
100g/ 3 oz  salami stick
2 x 250g / 8 oz  chicken breasts

  [1] group_name: Chicken spice mix:
      group_type: N/A
      ingredients (5):
1/2 tsp cooking salt / kosher salt
1/4 tsp black pepper
1/2 tsp garlic powder
3/4 tsp paprika
1/4 tsp sage powder

  [2] group_name: Creamy tomato parmesan risoni (orzo):
      group_type: N/A
      ingredients (13):
2  garlic cloves
1/2  onion
1/4 cup chardonnay or other dry white wine
1/4 cup tomato paste
1 1/4 cups risoni/orzo
400g / 14 oz  canned chickpeas
3 cups chicken stock
1/2 tsp cooking / kosher salt
1/4 tsp black pepper
3/4 cup thickened / heavy cream
1/3 cup parmesan
150g/ 5 oz  baby spinach
1/3 cup sun dried tomato

  [3] group_name: None ← NULL
      group_type: N/A
      ingredients (1):
2 tbsp roughly chopped basil

RECIPE

Recipes with multiple unnamed ingredient groups cannot be safely assumed as main, etc; these can be anyhting: garnish, etc. Can be called "Unknown".

In [6]:
# distribution of lenght of ingredients in each ingredient group

# count how many ingredients per group
group_lengths = Counter()
for r in recipes:
    for g in r["ingredient_groups"]:
        num_ingredients = len(g["ingredients"])
        group_lengths[num_ingredients] += 1

print("Distribution of ingredients per ingredient group:\n")
print(f"{'# Ingredients':<15} {'# Groups':<12} {'%'}")
print("-" * 35)
for length in sorted(group_lengths):
    count = group_lengths[length]
    total_groups = sum(group_lengths.values())
    pct = (count / total_groups) * 100
    bar = "█" * int(pct)
    print(f"{length:<15} {count:<12} {pct:.1f}%  {bar}")

print(f"\nTotal ingredient groups: {sum(group_lengths.values())}")
print(f"Mean ingredients per group: {sum(k * v for k, v in group_lengths.items()) / sum(group_lengths.values()):.1f}")
print(f"Range: {min(group_lengths)} – {max(group_lengths)}")

Distribution of ingredients per ingredient group:

# Ingredients   # Groups     %
-----------------------------------
0               5            0.1%  
1               356          9.3%  █████████
2               577          15.1%  ███████████████
3               492          12.9%  ████████████
4               459          12.0%  ████████████
5               379          9.9%  █████████
6               359          9.4%  █████████
7               309          8.1%  ████████
8               290          7.6%  ███████
9               187          4.9%  ████
10              121          3.2%  ███
11              93           2.4%  ██
12              76           2.0%  █
13              45           1.2%  █
14              30           0.8%  
15              24           0.6%  
16              10           0.3%  
17              6            0.2%  
18              2            0.1%  
19              3            0.1%  
20              1            0.0%  
27              1            0.

0 ingredient means that ingredient group should be deleted.

In [7]:
# unique ingredient group names
group_names = set()
for r in recipes:
    for g in r["ingredient_groups"]:
        group_names.add(g["group_name"])

print(f"Total unique ingredient group names: {len(group_names)}\n")
for name in sorted(group_names, key=lambda x: (x is None, str(x).lower())):
    print(f"  • {repr(name)}")

Total unique ingredient group names: 1788

  • '"Popcorn Salt":'
  • '* Asterisk-marked'
  • '1. Seafood Sauces for Lobster – choose:'
  • '2. Lobster Cocktail'
  • '3 layer vanilla cake'
  • '3. Lobster with Herb & Garlic Butter (enough for 2 lobsters)'
  • '4. Lobster with French Lemon Butter Sauce (2 lobsters)'
  • '5. Lobster with Salsa Verde (4 lobsters)'
  • '6. Lobster Rolls'
  • 'Add Ins'
  • 'Add ins (3 1/2 cups your choice, Note 3):'
  • 'Add Ins (Mix and Match!)'
  • 'Add ins (Note 1 for alternatives):'
  • 'Add ins – CHOOSE ONE:'
  • 'Add ins:'
  • 'Add Ins:'
  • 'Add-ins'
  • 'Add-ins (7 cups "anything" – Note 2)'
  • 'Add-Ins (Note 7):'
  • 'Add-ins (or use your own!):'
  • 'Add-ins – 1 1/2 cups “anything you want”, I used:'
  • 'Add-ins:'
  • 'Almond Coating (optional, Note 5):'
  • 'Almond decoration:'
  • 'Almond Sauce'
  • 'Alternative add in options'
  • 'Alternative stir in'
  • 'AMATRICE SPAGHETTI (SPAGHETTI ALL’AMATRICIANA):'
  • 'Apple Cider Drizzle'
  • 'Apple F

In [8]:
# unique instruction group names
group_names = set()
for r in recipes:
    for g in r["instruction_groups"]:
        group_names.add(g["group_name"])

print(f"Total unique instruction group names: {len(group_names)}\n")
for name in sorted(group_names, key=lambda x: (x is None, str(x).lower())):
    print(f"  • {repr(name)}")

Total unique instruction group names: 1239

  • ''
  • '!Mustard Sauce'
  • '"Holiday Tapenade":'
  • '"Popcorn Salt"'
  • '(ABBREVIATED RECIPE)'
  • '(FULL RECIPE)'
  • '.'
  • '3 Minute Festive “Baked” Brie:'
  • '<br>'
  • 'ABBREVIATED'
  • 'Abbreviated directions:'
  • 'Abbreviated Recipe'
  • 'ABBREVIATED RECIPE'
  • 'Abbreviated recipe'
  • 'Abbreviated recipe (hand-mixed):'
  • 'ABBREVIATED RECIPE:'
  • 'Abbreviated recipe:'
  • 'Abbreviated steps for baking pros:'
  • 'ABBREVIATED:'
  • 'Additional Flavouring Suggestions'
  • 'Aioli:'
  • 'Alfredo Sauce:'
  • 'All Dressings except Caesar:'
  • 'All sauces except Thai Sauce:'
  • 'Almond bits:'
  • 'Almond Sauce'
  • 'AMATRICE SPAGHETTI (SPAGHETTI ALL’AMATRICIANA):'
  • 'Amount to Use (Note 6):'
  • 'Apple Cider Drizzle'
  • 'Apple Pie Filling:'
  • 'Apple:'
  • 'Apples:'
  • 'Aromatics'
  • 'Assemble'
  • 'Assemble & Bake'
  • 'Assemble & bake:'
  • 'Assemble & Bake:'
  • 'Assemble and Bake'
  • 'Assemble And Bake Quiche'
  • '

In [9]:
# compare number of ingredient groups and instruction groups per recipe
mismatches = 0
same = 0
ing_more = 0
inst_more = 0

for r in recipes:
    ing_count = len(r["ingredient_groups"])
    inst_count = len(r["instruction_groups"])
    
    if ing_count != inst_count:
        mismatches += 1
        marker = "mismatch"
    else:
        same += 1
        marker = ""
    
    if ing_count > inst_count:
        ing_more += 1
    elif inst_count > ing_count:
        inst_more += 1
    
print(f"  Same count:                   {same}")
print(f"  Mismatched:                   {mismatches}")
print(f"  More ingredient groups:       {ing_more}")
print(f"  More instruction groups:      {inst_more}")

  Same count:                   464
  Mismatched:                   1018
  More ingredient groups:       789
  More instruction groups:      229


Not a one-to-one relation between ingredient group and instruction group.

In [10]:
def clean_instruction_group_name(name):
    """Normalise instruction group name."""
    if name is None or not isinstance(name, str):
        return name
    
    original = name
    
    # 1. Strip wrapping parentheses: (FULL RECIPE) -> FULL RECIPE
    if name.startswith("(") and name.endswith(")"):
        name = name[1:-1]
    
    # 2. Strip trailing characters: colons, quotes, exclamation, dots, whitespace
    name = name.rstrip(':\'"!., ')
    
    # 3. Strip leading characters: quotes, exclamation, whitespace
    name = name.lstrip('\'"!., ')
    
    # 4. Remove stray quotes inside the name
    name = name.replace('"', '').replace('“', '').replace('”', '').replace("'", "")
    
    # 5. Collapse multiple spaces
    name = re.sub(r'\s+', ' ', name).strip()
    
    # 6. Title case
    name = name.title()
    
    return name if name else original  # If we stripped everything, keep original

# Process and track changes
changes = []

for recipe in recipes:
    for group in recipe["instruction_groups"]:
        old_name = group["group_name"]
        new_name = clean_instruction_group_name(old_name)
        
        if old_name != new_name:
            changes.append((old_name, new_name, recipe["recipe_name"]))
        
        group["group_name"] = new_name

# Save
with open("recipes_eda.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

In [11]:
with open("recipes_eda.json", "r") as f:
    recipes = json.load(f)

group_names = set()
for r in recipes:
    for g in r["instruction_groups"]:
        group_names.add(g["group_name"])

print(f"Total unique instruction group names: {len(group_names)}\n")
for name in sorted(group_names, key=lambda x: (x is None, str(x).lower())):
    print(f"  • {repr(name)}")

Total unique instruction group names: 1114

  • ''
  • '.'
  • '3 Minute Festive Baked Brie'
  • '<Br>'
  • 'Abbreviated'
  • 'Abbreviated Directions'
  • 'Abbreviated Recipe'
  • 'Abbreviated Recipe (Hand-Mixed)'
  • 'Abbreviated Steps For Baking Pros'
  • 'Additional Flavouring Suggestions'
  • 'Aioli'
  • 'Alfredo Sauce'
  • 'All Dressings Except Caesar'
  • 'All Sauces Except Thai Sauce'
  • 'Almond Bits'
  • 'Almond Sauce'
  • 'Amatrice Spaghetti (Spaghetti All’Amatriciana)'
  • 'Amount To Use (Note 6)'
  • 'Apple'
  • 'Apple Cider Drizzle'
  • 'Apple Pie Filling'
  • 'Apples'
  • 'Aromatics'
  • 'Assemble'
  • 'Assemble & Bake'
  • 'Assemble And Bake'
  • 'Assemble And Bake Quiche'
  • 'Assemble Baklava (See Layers Visual Below)'
  • 'Assemble Bowls'
  • 'Assemble Burgers'
  • 'Assemble Burritos'
  • 'Assemble Each Of The Meals As Per The Recipe Instructions. Ensure You Follow The Instructions For Making The Recipes To Freeze'
  • 'Assemble Flan Pâtissier'
  • 'Assemble For Pan F

In [12]:
# distribution of isntruction gorups per recipe
with open("recipes_eda.json", "r") as f:
    recipes = json.load(f)

# Count instruction groups per recipe
group_counts = Counter()
for r in recipes:
    num_groups = len(r["instruction_groups"])
    group_counts[num_groups] += 1

print("Distribution of instruction groups per recipe:\n")
print(f"{'# Groups':<10} {'# Recipes':<12} {'%'}")
print("-" * 30)
for num_groups in sorted(group_counts):
    count = group_counts[num_groups]
    pct = (count / len(recipes)) * 100
    bar = "█" * int(pct / 2)
    print(f"{num_groups:<10} {count:<12} {pct:.1f}%  {bar}")

print(f"\nTotal recipes: {len(recipes)}")
print(f"Mean groups per recipe: {sum(k * v for k, v in group_counts.items()) / len(recipes):.1f}")
print(f"Range: {min(group_counts)} – {max(group_counts)}")

Distribution of instruction groups per recipe:

# Groups   # Recipes    %
------------------------------
1          786          53.0%  ██████████████████████████
2          302          20.4%  ██████████
3          165          11.1%  █████
4          134          9.0%  ████
5          54           3.6%  █
6          18           1.2%  
7          13           0.9%  
8          3            0.2%  
9          4            0.3%  
10         1            0.1%  
12         1            0.1%  
13         1            0.1%  

Total recipes: 1482
Mean groups per recipe: 2.0
Range: 1 – 13


In [13]:
# how many of single isntruction group per recipe have group name None, or '', or '.', or '<br>', ' '
junk_values = {None, '', '.', '<br>', ' '}

single_group_recipes = []
single_group_junk = 0
single_group_named = 0

for r in recipes:
    groups = r["instruction_groups"]
    
    if len(groups) == 1:
        single_group_recipes.append(r)
        
        if groups[0]["group_name"] in junk_values:
            single_group_junk += 1
        else:
            single_group_named += 1

print(f"Recipes with exactly 1 instruction group: {len(single_group_recipes)}")
print(f"  Group name is junk (None, '', '.', '<br>', ' '):  {single_group_junk}")
print(f"  Group name is valid:                              {single_group_named}")

# Breakdown of junk values
junk_counts = Counter()
for r in single_group_recipes:
    name = r["instruction_groups"][0]["group_name"]
    if name in junk_values:
        junk_counts[repr(name)] += 1

print(f"\nJunk value breakdown:")
for val, count in junk_counts.items():
    print(f"  {val}: {count}")

Recipes with exactly 1 instruction group: 786
  Group name is junk (None, '', '.', '<br>', ' '):  781
  Group name is valid:                              5

Junk value breakdown:
  None: 781


Single instruction group with junk group name can be replaced to Full Recipe.

In [14]:
with open("recipes_eda.json", "r") as f:
    recipes = json.load(f)

junk_values = {None, '', '.', '<br>', ' '}
changed = 0

for r in recipes:
    groups = r["instruction_groups"]
    
    if len(groups) == 1 and groups[0]["group_name"] in junk_values:
        old = groups[0]["group_name"]
        groups[0]["group_name"] = "Full Recipe"
        changed += 1
        print(f"  {repr(old)} → 'Full Recipe'  |  {r['recipe_name']}")

# Save
with open("recipes_eda.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

print(f"\nChanged {changed} instruction group names to 'Full Recipe'")

  None → 'Full Recipe'  |  Nagi’s Salad Sandwich
  None → 'Full Recipe'  |  Instant Pink Taco Sauce
  None → 'Full Recipe'  |  Creamy Jalapeño Sauce
  None → 'Full Recipe'  |  Baharat (aka Lebanese 7 Spice)
  None → 'Full Recipe'  |  Creamy French Dressing
  None → 'Full Recipe'  |  Herbes de Provence
  None → 'Full Recipe'  |  Tzatziki
  None → 'Full Recipe'  |  Chimichurri butter
  None → 'Full Recipe'  |  Thai Dipping Sauce – Nam Jim Jaew
  None → 'Full Recipe'  |  Thai Grilled Chicken (Gai Yang)
  None → 'Full Recipe'  |  White Christmas Rocky Road
  None → 'Full Recipe'  |  Raspberry Coulis
  None → 'Full Recipe'  |  Lemon Curd
  None → 'Full Recipe'  |  Lemon Yogurt Sauce
  None → 'Full Recipe'  |  Homemade Peri-naise
  None → 'Full Recipe'  |  Lemongrass Chicken Rice Paper Rolls
  None → 'Full Recipe'  |  Korean Beef Bulgogi Rice Bowls – the easy way
  None → 'Full Recipe'  |  Broccoli Pearl Crunch Salad
  None → 'Full Recipe'  |  Pikelets
  None → 'Full Recipe'  |  Whipped rico

In [15]:
# number instruction and ingredient group names that are the same 

with open("recipes_eda.json", "r") as f:
    recipes = json.load(f)

# Collect unique names from both
ingredient_names = set()
instruction_names = set()

for r in recipes:
    for g in r["ingredient_groups"]:
        name = g["group_name"]
        if name is not None:
            ingredient_names.add(name)
    for g in r["instruction_groups"]:
        name = g["group_name"]
        if name is not None:
            instruction_names.add(name)

# Find overlap
overlap = ingredient_names & instruction_names

print(f"Unique ingredient group names:  {len(ingredient_names)}")
print(f"Unique instruction group names:  {len(instruction_names)}")
print(f"Names in both:                  {len(overlap)}\n")

Unique ingredient group names:  1787
Unique instruction group names:  1113
Names in both:                  164



Further shows that there is no clear relationship between ingredient and instruction group names.

In [16]:
# distribution of junk inst group names per recipe, in relation to number of inst groups per recipe
with open("recipes_eda.json", "r") as f:
    recipes = json.load(f)

junk_values = {None, '', '.', '<br>', ' '}

# Count recipes by (total groups, junk groups)
distribution = Counter()
junk_recipes = []

for r in recipes:
    groups = r["instruction_groups"]
    total = len(groups)
    junk_count = sum(1 for g in groups if g["group_name"] in junk_values)
    
    distribution[(total, junk_count)] += 1
    
    if junk_count > 0:
        junk_recipes.append((r["recipe_name"], total, junk_count))

print("Distribution of junk instruction group names:\n")
print(f"{'Total Groups':<14} {'Junk Groups':<13} {'# Recipes':<12} {'%'}")
print("-" * 50)

for (total, junk), count in sorted(distribution.items()):
    if junk > 0:
        pct = (count / len(recipes)) * 100
        bar = "█" * int(pct * 5)
        print(f"{total:<14} {junk:<13} {count:<12} {pct:.1f}%  {bar}")

print(f"\nRecipes with junk groups: {len(junk_recipes)}")
print(f"Total recipes: {len(recipes)}")

# Show the ones with multiple junk groups
multi_junk = [(name, total, junk) for name, total, junk in junk_recipes if junk >= 2]
if multi_junk:
    print(f"\nRecipes with 2+ junk groups ({len(multi_junk)}):")
    for name, total, junk in multi_junk:
        print(f"  {junk}/{total} junk  |  {name}")

Distribution of junk instruction group names:

Total Groups   Junk Groups   # Recipes    %
--------------------------------------------------
2              1             159          10.7%  █████████████████████████████████████████████████████
2              2             7            0.5%  ██
3              1             72           4.9%  ████████████████████████
3              2             3            0.2%  █
3              3             2            0.1%  
4              1             52           3.5%  █████████████████
4              2             3            0.2%  █
5              1             15           1.0%  █████
5              2             2            0.1%  
6              1             5            0.3%  █
7              1             3            0.2%  █
7              2             1            0.1%  
8              1             2            0.1%  
9              1             1            0.1%  
10             10            1            0.1%  
12             2 

In [17]:
# for recipes with multpile instruction groups, with single or multipe junk group names
# this can be replaced to "Other Component" (Same as saying all junk group names can be 
# replaced to "Other Component"). 
# This does mean that instruction groups will not have unique names for each respective group.

import json

with open("recipes_eda.json", "r") as f:
    recipes = json.load(f)

junk_values = {None, '', '.', '<br>', ' ', '<Br>'}
changed = 0

for r in recipes:
    groups = r["instruction_groups"]
    
    for g in groups:
        if g["group_name"] in junk_values:
            old = repr(g["group_name"])
            g["group_name"] = "Other Component"
            changed += 1
# Save
with open("recipes_eda.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

print(f"\nChanged {changed} junk instruction group names to 'Other Component'")


Changed 361 junk instruction group names to 'Other Component'


In [18]:
# print unique instruction group names

group_names = set()
for r in recipes:
    for g in r["instruction_groups"]:
        group_names.add(g["group_name"])

print(f"Total unique instruction group names: {len(group_names)}\n")
for name in sorted(group_names, key=lambda x: (x is None, str(x).lower())):
    print(f"  • {repr(name)}")

Total unique instruction group names: 1111

  • '3 Minute Festive Baked Brie'
  • 'Abbreviated'
  • 'Abbreviated Directions'
  • 'Abbreviated Recipe'
  • 'Abbreviated Recipe (Hand-Mixed)'
  • 'Abbreviated Steps For Baking Pros'
  • 'Additional Flavouring Suggestions'
  • 'Aioli'
  • 'Alfredo Sauce'
  • 'All Dressings Except Caesar'
  • 'All Sauces Except Thai Sauce'
  • 'Almond Bits'
  • 'Almond Sauce'
  • 'Amatrice Spaghetti (Spaghetti All’Amatriciana)'
  • 'Amount To Use (Note 6)'
  • 'Apple'
  • 'Apple Cider Drizzle'
  • 'Apple Pie Filling'
  • 'Apples'
  • 'Aromatics'
  • 'Assemble'
  • 'Assemble & Bake'
  • 'Assemble And Bake'
  • 'Assemble And Bake Quiche'
  • 'Assemble Baklava (See Layers Visual Below)'
  • 'Assemble Bowls'
  • 'Assemble Burgers'
  • 'Assemble Burritos'
  • 'Assemble Each Of The Meals As Per The Recipe Instructions. Ensure You Follow The Instructions For Making The Recipes To Freeze'
  • 'Assemble Flan Pâtissier'
  • 'Assemble For Pan Frying'
  • 'Assemble Gyros

Overall, for both inst and ing group names, their reasoning is not always consistent.

Abbreviated directions can stay for now, treat this as "augmented" data for training.

In [19]:
# check for any blank instructions in instruction groups
with open("recipes_eda.json", "r") as f:
    recipes = json.load(f)

empty_instructions = []

for r in recipes:
    for g in r["instruction_groups"]:
        if len(g["instructions"]) == 0:
            empty_instructions.append({
                "recipe": r["recipe_name"],
                "group_name": g["group_name"],
                "total_groups_in_recipe": len(r["instruction_groups"])
            })
        else:
            # Also check for individual blank steps
            for i, step in enumerate(g["instructions"]):
                if not step or step.strip() == '':
                    empty_instructions.append({
                        "recipe": r["recipe_name"],
                        "group_name": g["group_name"],
                        "step_index": i,
                        "issue": "blank step"
                    })

print(f"Found {len(empty_instructions)} issues:\n")

for item in empty_instructions:
    if "step_index" in item:
        print(f"  Recipe: {item['recipe']}")
        print(f"  Group:  {item['group_name']}")
        print(f"  Step #{item['step_index']} is blank")
    else:
        print(f"  Recipe: {item['recipe']}")
        print(f"  Group:  {item['group_name']}")
        print(f"  Empty instructions list (0 steps)")
    print()

Found 30 issues:

  Recipe: Porchetta – Dozer’s last recipe
  Group:  Full Directions
  Empty instructions list (0 steps)

  Recipe: JB’s Banana Bread
  Group:  Full Recipe
  Empty instructions list (0 steps)

  Recipe: JB’s French Pork Schnitzel
  Group:  Full Recipe
  Empty instructions list (0 steps)

  Recipe: Salted Caramel Cheesecake
  Group:  Full Recipe
  Empty instructions list (0 steps)

  Recipe: Green Ginger Rice Noodles
  Group:  Full Recipe
  Empty instructions list (0 steps)

  Recipe: JB’s Pommes Anna
  Group:  Full Recipe
  Empty instructions list (0 steps)

  Recipe: Pavlova Bombs
  Group:  Full Recipe
  Empty instructions list (0 steps)

  Recipe: Coquilles Saint-Jacques
  Group:  Full Recipe
  Empty instructions list (0 steps)

  Recipe: Birria Tacos
  Group:  Full Recipe
  Empty instructions list (0 steps)

  Recipe: Vanilla Cream Filled Chocolate Ganache Cupcakes
  Group:  Full Recipe
  Empty instructions list (0 steps)

  Recipe: Smothered rissoles
  Group:  Full

In [20]:
# distribution of number of lines in instructions 

# Count steps per instruction group
step_counts = Counter()
for r in recipes:
    for g in r["instruction_groups"]:
        num_steps = len(g["instructions"])
        step_counts[num_steps] += 1

print("Distribution of steps per instruction group:\n")
print(f"{'# Steps':<10} {'# Groups':<12} {'%'}")
print("-" * 30)
for num_steps in sorted(step_counts):
    count = step_counts[num_steps]
    total_groups = sum(step_counts.values())
    pct = (count / total_groups) * 100
    bar = "█" * int(pct / 2)
    print(f"{num_steps:<10} {count:<12} {pct:.1f}%  {bar}")

print(f"\nTotal instruction groups: {sum(step_counts.values())}")
print(f"Mean steps per group: {sum(k * v for k, v in step_counts.items()) / sum(step_counts.values()):.1f}")
print(f"Range: {min(step_counts)} – {max(step_counts)}")

Distribution of steps per instruction group:

# Steps    # Groups     %
------------------------------
0          30           1.0%  
1          416          13.9%  ██████
2          415          13.9%  ██████
3          429          14.4%  ███████
4          336          11.3%  █████
5          282          9.4%  ████
6          270          9.0%  ████
7          251          8.4%  ████
8          212          7.1%  ███
9          136          4.6%  ██
10         83           2.8%  █
11         57           1.9%  
12         34           1.1%  
13         20           0.7%  
14         9            0.3%  
15         3            0.1%  
17         1            0.0%  
18         1            0.0%  

Total instruction groups: 2985
Mean steps per group: 4.7
Range: 0 – 18


In [22]:
# number of characters in instruction distriubtion

# Count total characters per instruction group
char_counts = Counter()
for r in recipes:
    for g in r["instruction_groups"]:
        total_chars = sum(len(step) for step in g["instructions"])
        # Bucket into ranges for readability
        if total_chars == 0:
            bucket = 0
        elif total_chars <= 50:
            bucket = "1–50"
        elif total_chars <= 100:
            bucket = "51–100"
        elif total_chars <= 200:
            bucket = "101–200"
        elif total_chars <= 300:
            bucket = "201–300"
        elif total_chars <= 500:
            bucket = "301–500"
        elif total_chars <= 750:
            bucket = "501–750"
        elif total_chars <= 1000:
            bucket = "751–1000"
        elif total_chars <= 1500:
            bucket = "1001–1500"
        elif total_chars <= 2000:
            bucket = "1501–2000"
        else:
            bucket = "2001+"
        
        char_counts[bucket] += 1

print("Distribution of total characters per instruction group:\n")
print(f"{'Char Range':<15} {'# Groups':<12} {'%'}")
print("-" * 35)

bucket_order = [0, "1–50", "51–100", "101–200", "201–300", "301–500", "501–750", "751–1000", "1001–1500", "1501–2000", "2001+"]
for bucket in bucket_order:
    count = char_counts.get(bucket, 0)
    total_groups = sum(char_counts.values())
    pct = (count / total_groups) * 100 if total_groups > 0 else 0
    bar = "█" * int(pct / 2)
    print(f"{str(bucket):<15} {count:<12} {pct:.1f}%  {bar}")

# Overall stats
all_chars = []
for r in recipes:
    for g in r["instruction_groups"]:
        all_chars.append(sum(len(step) for step in g["instructions"]))

print(f"\nTotal instruction groups: {len(all_chars)}")
print(f"Mean characters per group: {sum(all_chars) / len(all_chars):.0f}")
print(f"Min characters: {min(all_chars)}")
print(f"Max characters: {max(all_chars)}")

Distribution of total characters per instruction group:

Char Range      # Groups     %
-----------------------------------
0               30           1.0%  
1–50            36           1.2%  
51–100          96           3.2%  █
101–200         308          10.3%  █████
201–300         405          13.6%  ██████
301–500         606          20.3%  ██████████
501–750         581          19.5%  █████████
751–1000        431          14.4%  ███████
1001–1500       408          13.7%  ██████
1501–2000       71           2.4%  █
2001+           13           0.4%  

Total instruction groups: 2985
Mean characters per group: 588
Min characters: 0
Max characters: 2681


In [23]:
# remove instruction group with no instruction
removed = 0

for r in recipes:
    before = len(r["instruction_groups"])
    r["instruction_groups"] = [g for g in r["instruction_groups"] if len(g["instructions"]) > 0]
    after = len(r["instruction_groups"])
    
    if before != after:
        removed += (before - after)
        print(f"  Removed {before - after} empty group(s)  |  {r['recipe_name']}")

# Save
with open("recipes_eda.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

print(f"\nRemoved {removed} empty instruction groups")

  Removed 1 empty group(s)  |  Porchetta – Dozer’s last recipe
  Removed 1 empty group(s)  |  JB’s Banana Bread
  Removed 1 empty group(s)  |  JB’s French Pork Schnitzel
  Removed 1 empty group(s)  |  Salted Caramel Cheesecake
  Removed 1 empty group(s)  |  Green Ginger Rice Noodles
  Removed 1 empty group(s)  |  JB’s Pommes Anna
  Removed 1 empty group(s)  |  Pavlova Bombs
  Removed 1 empty group(s)  |  Coquilles Saint-Jacques
  Removed 1 empty group(s)  |  Birria Tacos
  Removed 1 empty group(s)  |  Vanilla Cream Filled Chocolate Ganache Cupcakes
  Removed 1 empty group(s)  |  Smothered rissoles
  Removed 1 empty group(s)  |  Tomahawk Steak
  Removed 1 empty group(s)  |  Popcorn chicken – better than KFC!
  Removed 1 empty group(s)  |  Prawn cocktail mini tacos (shrimp)
  Removed 1 empty group(s)  |  Crunchy chicken taco fingers (baked!)
  Removed 1 empty group(s)  |  Thai red duck curry
  Removed 1 empty group(s)  |  Devilled eggs
  Removed 1 empty group(s)  |  The Chocolate Chip Co

In [24]:
# distribution of number of lines in instructions 

# Count steps per instruction group
step_counts = Counter()
for r in recipes:
    for g in r["instruction_groups"]:
        num_steps = len(g["instructions"])
        step_counts[num_steps] += 1

print("Distribution of steps per instruction group:\n")
print(f"{'# Steps':<10} {'# Groups':<12} {'%'}")
print("-" * 30)
for num_steps in sorted(step_counts):
    count = step_counts[num_steps]
    total_groups = sum(step_counts.values())
    pct = (count / total_groups) * 100
    bar = "█" * int(pct / 2)
    print(f"{num_steps:<10} {count:<12} {pct:.1f}%  {bar}")

print(f"\nTotal instruction groups: {sum(step_counts.values())}")
print(f"Mean steps per group: {sum(k * v for k, v in step_counts.items()) / sum(step_counts.values()):.1f}")
print(f"Range: {min(step_counts)} – {max(step_counts)}")

# number of characters in instruction distriubtion

# Count total characters per instruction group
char_counts = Counter()
for r in recipes:
    for g in r["instruction_groups"]:
        total_chars = sum(len(step) for step in g["instructions"])
        # Bucket into ranges for readability
        if total_chars == 0:
            bucket = 0
        elif total_chars <= 50:
            bucket = "1–50"
        elif total_chars <= 100:
            bucket = "51–100"
        elif total_chars <= 200:
            bucket = "101–200"
        elif total_chars <= 300:
            bucket = "201–300"
        elif total_chars <= 500:
            bucket = "301–500"
        elif total_chars <= 750:
            bucket = "501–750"
        elif total_chars <= 1000:
            bucket = "751–1000"
        elif total_chars <= 1500:
            bucket = "1001–1500"
        elif total_chars <= 2000:
            bucket = "1501–2000"
        else:
            bucket = "2001+"
        
        char_counts[bucket] += 1

print("Distribution of total characters per instruction group:\n")
print(f"{'Char Range':<15} {'# Groups':<12} {'%'}")
print("-" * 35)

bucket_order = [0, "1–50", "51–100", "101–200", "201–300", "301–500", "501–750", "751–1000", "1001–1500", "1501–2000", "2001+"]
for bucket in bucket_order:
    count = char_counts.get(bucket, 0)
    total_groups = sum(char_counts.values())
    pct = (count / total_groups) * 100 if total_groups > 0 else 0
    bar = "█" * int(pct / 2)
    print(f"{str(bucket):<15} {count:<12} {pct:.1f}%  {bar}")

# Overall stats
all_chars = []
for r in recipes:
    for g in r["instruction_groups"]:
        all_chars.append(sum(len(step) for step in g["instructions"]))

print(f"\nTotal instruction groups: {len(all_chars)}")
print(f"Mean characters per group: {sum(all_chars) / len(all_chars):.0f}")
print(f"Min characters: {min(all_chars)}")
print(f"Max characters: {max(all_chars)}")

Distribution of steps per instruction group:

# Steps    # Groups     %
------------------------------
1          416          14.1%  ███████
2          415          14.0%  ███████
3          429          14.5%  ███████
4          336          11.4%  █████
5          282          9.5%  ████
6          270          9.1%  ████
7          251          8.5%  ████
8          212          7.2%  ███
9          136          4.6%  ██
10         83           2.8%  █
11         57           1.9%  
12         34           1.2%  
13         20           0.7%  
14         9            0.3%  
15         3            0.1%  
17         1            0.0%  
18         1            0.0%  

Total instruction groups: 2955
Mean steps per group: 4.7
Range: 1 – 18
Distribution of total characters per instruction group:

Char Range      # Groups     %
-----------------------------------
0               0            0.0%  
1–50            36           1.2%  
51–100          96           3.2%  █
101–200         3

In [25]:
# going back to ingredients, let's see all recipes which ingredient groups that have 0 ingredients
with open("recipes_eda.json", "r") as f:
    recipes = json.load(f)

empty_groups = []

for r in recipes:
    for g in r["ingredient_groups"]:
        if len(g["ingredients"]) == 0:
            empty_groups.append({
                "recipe": r["recipe_name"],
                "group_name": g["group_name"],
                "total_groups_in_recipe": len(r["ingredient_groups"])
            })

print(f"Found {len(empty_groups)} ingredient groups with 0 ingredients:\n")

for item in empty_groups:
    print(f"  Recipe: {item['recipe']}")
    print(f"  Group Name:  {item['group_name']}")
    print(f"  Total groups in recipe: {item['total_groups_in_recipe']}")
    print()


Found 5 ingredient groups with 0 ingredients:

  Recipe: Vegetable Samosa Pie
  Group Name:  * Asterisk-marked
  Total groups in recipe: 5

  Recipe: Gozleme!!
  Group Name:  Choose ONE Filling (Note 1)
  Total groups in recipe: 5

  Recipe: Quesadilla (Beef, Vegetable or Chicken)
  Group Name:  CHOOSE ONE FILLING
  Total groups in recipe: 6

  Recipe: Peanut Sauce for Stir Fries
  Group Name:  TO USE THE SAUCE:
  Total groups in recipe: 5

  Recipe: Rice recipes - rice meal recipes for dinner
  Group Name:  None
  Total groups in recipe: 3



In [26]:
# remove ingredient groups wiht 0 ingredients

with open("recipes_eda.json", "r") as f:
    recipes = json.load(f)

removed = 0

for r in recipes:
    before = len(r["ingredient_groups"])
    r["ingredient_groups"] = [g for g in r["ingredient_groups"] if len(g["ingredients"]) > 0]
    after = len(r["ingredient_groups"])
    
    if before != after:
        removed += (before - after)
        print(f"  Removed {before - after} empty group(s)  |  {r['recipe_name']}")

# Save
with open("recipes_eda.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

print(f"\nRemoved {removed} empty ingredient groups")

  Removed 1 empty group(s)  |  Vegetable Samosa Pie
  Removed 1 empty group(s)  |  Gozleme!!
  Removed 1 empty group(s)  |  Quesadilla (Beef, Vegetable or Chicken)
  Removed 1 empty group(s)  |  Peanut Sauce for Stir Fries
  Removed 1 empty group(s)  |  Rice recipes - rice meal recipes for dinner

Removed 5 empty ingredient groups


In [27]:
# number instruction and ingredient group names that are the same 

with open("recipes_eda.json", "r") as f:
    recipes = json.load(f)

# Collect unique names from both
ingredient_names = set()
instruction_names = set()

for r in recipes:
    for g in r["ingredient_groups"]:
        name = g["group_name"]
        if name is not None:
            ingredient_names.add(name)
    for g in r["instruction_groups"]:
        name = g["group_name"]
        if name is not None:
            instruction_names.add(name)

# Find overlap
overlap = ingredient_names & instruction_names

print(f"Unique ingredient group names:  {len(ingredient_names)}")
print(f"Unique instruction group names:  {len(instruction_names)}")
print(f"Names in both:                  {len(overlap)}\n")

Unique ingredient group names:  1783
Unique instruction group names:  1110
Names in both:                  164



In [28]:
import json

with open("recipes_eda.json", "r") as f:
    recipes = json.load(f)

print("Comparing ingredient and instruction group names within each recipe:\n")

exact_matches = 0
partial_matches = 0
no_relation = 0

for r in recipes:
    ing_names = set(g["group_name"].lower() for g in r["ingredient_groups"] if g["group_name"])
    inst_names = set(g["group_name"].lower() for g in r["instruction_groups"] if g["group_name"])
    
    if not ing_names or not inst_names:
        continue
    
    # Check for exact matches
    exact = ing_names & inst_names
    
    # Check for partial matches (one contains the other)
    partial = set()
    for ing in ing_names:
        for inst in inst_names:
            if ing != inst and (ing in inst or inst in ing):
                partial.add((ing, inst))
    
    if exact or partial:
        print(f"Recipe: {r['recipe_name']}")
        if exact:
            print(f"  Exact matches: {exact}")
            exact_matches += 1
        if partial:
            for ing, inst in partial:
                print(f"  Partial: '{ing}' ↔ '{inst}'")
            partial_matches += 1
        print()
    else:
        no_relation += 1

print(f"\nSummary:")
print(f"  Recipes with exact name matches:    {exact_matches}")
print(f"  Recipes with partial name matches:  {partial_matches}")
print(f"  Recipes with no relation:           {no_relation}")

Comparing ingredient and instruction group names within each recipe:

Recipe: Birthday Cake
  Partial: 'my favourite chocolate frosting' ↔ 'chocolate frosting'

Recipe: Poulet Rôti – French Roast Chicken
  Partial: 'potatoes' ↔ 'prepare chicken and potatoes'

Recipe: Pastel de Choclo (Chilean Beef and Corn Casserole)
  Partial: 'corn layer:' ↔ 'corn layer'
  Partial: 'beef filling:' ↔ 'beef filling'

Recipe: 12 Minute Baked Salmon Fajitas
  Partial: 'serving:' ↔ 'serving'

Recipe: Porchetta – Dozer’s last recipe
  Partial: 'gravy:' ↔ 'gravy'

Recipe: Layered Golden Rice Bake with Spiced Meat
  Partial: 'golden rice:' ↔ 'golden rice'

Recipe: Baby Gem Lettuce Salad
  Partial: 'croutons:' ↔ 'croutons'

Recipe: JB’s French Pork Schnitzel
  Partial: 'crumbing and cooking:' ↔ 'crumbing and cooking'

Recipe: Salted Caramel Cheesecake
  Partial: 'cheesecake filling:' ↔ 'cheesecake filling'

Recipe: Roasted Garlic Butter
  Partial: 'roasted garlic:' ↔ 'roasted garlic'

Recipe: Pavlova Bombs
  

There may be semantic similarities, which can be assessed via an LLM model.

In [29]:
import json
import requests

with open("recipes_eda.json", "r") as f:
    recipes = json.load(f)

# Build pairs of (ingredient_names, instruction_names) per recipe
recipe_pairs = []
for r in recipes:
    ing_names = [g["group_name"] for g in r["ingredient_groups"] if g["group_name"]]
    inst_names = [g["group_name"] for g in r["instruction_groups"] if g["group_name"]]
    
    if ing_names and inst_names:
        recipe_pairs.append({
            "recipe": r["recipe_name"],
            "ingredient_groups": ing_names,
            "instruction_groups": inst_names
        })

print(f"Analyzing {len(recipe_pairs)} recipes for semantic relationships...\n")

# Build prompt with all pairs
pairs_text = ""
for i, rp in enumerate(recipe_pairs):
    pairs_text += f"Recipe {i}: \"{rp['recipe']}\"\n"
    pairs_text += f"  Ingredient groups: {json.dumps(rp['ingredient_groups'])}\n"
    pairs_text += f"  Instruction groups: {json.dumps(rp['instruction_groups'])}\n\n"

prompt = f"""I have {len(recipe_pairs)} recipes. For each recipe, I have a list of ingredient group names and instruction group names.

Identify pairs where an ingredient group and instruction group refer to the SAME component or step. 

Examples of matches:
- "Marinade" (ingredient) ↔ "Marinate Chicken" (instruction) — same thing
- "Crostinis" ↔ "Crostini for Carpaccio" — same component
- "Almond Sauce" ↔ "Almond Sauce" — identical
- "Assemble" ↔ "Assembling" — same action
- "Bechamel Sauce" ↔ "Make Bechamel Sauce" — same component
- "Full Recipe" ↔ "Full Recipe" — identical

Examples of non-matches:
- "Main" ↔ "Full Recipe" — different things
- "Serving" ↔ "Cook Pasta" — different things

Return a JSON object with this structure:
{{
  "matches": [
    {{
      "recipe": "Recipe Name",
      "ingredient_group": "Marinade",
      "instruction_group": "Marinate Chicken",
      "relationship": "same component"
    }}
  ]
}}

Only include pairs that ARE related. Skip recipes with no relationships.

Here are the recipes:
{pairs_text}"""

# Call Ollama
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.1:8b",
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {"temperature": 0}
    }
)

result = json.loads(response.json()["response"])

# Print results
matches = result.get("matches", [])
print(f"Found {len(matches)} semantic relationships:\n")

# Group by relationship type
from collections import Counter
rel_types = Counter(m["relationship"] for m in matches)
print("Relationship types:")
for rel, count in rel_types.items():
    print(f"  {rel}: {count}")

print(f"\nSample matches:")
for m in matches[:20]:
    print(f"  {m['recipe']}")
    print(f"    ingredient: {m['ingredient_group']}")
    print(f"    instruction: {m['instruction_group']}")
    print(f"    relationship: {m['relationship']}")
    print()

# Save
with open("ingredient_instruction_relations.json", "w") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print("Saved to ingredient_instruction_relations.json")

Analyzing 1198 recipes for semantic relationships...

Found 0 semantic relationships:

Relationship types:

Sample matches:
Saved to ingredient_instruction_relations.json


In [31]:
# relation of number of ingredient and instruction groups
with open("recipes_eda.json", "r") as f:
    recipes = json.load(f)

# Count recipes by (ing_groups, inst_groups)
pair_counts = Counter()
for r in recipes:
    ing = len(r["ingredient_groups"])
    inst = len(r["instruction_groups"])
    pair_counts[(ing, inst)] += 1

print("Relation between # ingredient groups and # instruction groups:\n")
print(f"{'Ing Groups':<12} {'Inst Groups':<13} {'# Recipes':<12} {'%'}")
print("-" * 50)

for (ing, inst), count in sorted(pair_counts.items()):
    pct = (count / len(recipes)) * 100
    bar = "█" * int(pct * 2)
    print(f"{ing:<12} {inst:<13} {count:<12} {pct:.1f}%  {bar}")

print(f"\nTotal recipes: {len(recipes)}")

# Summary stats
match = sum(1 for r in recipes if len(r["ingredient_groups"]) == len(r["instruction_groups"]))
ing_more = sum(1 for r in recipes if len(r["ingredient_groups"]) > len(r["instruction_groups"]))
inst_more = sum(1 for r in recipes if len(r["ingredient_groups"]) < len(r["instruction_groups"]))

print(f"\nSame count:  {match} ({match/len(recipes)*100:.1f}%)")
print(f"Ing > Inst:  {ing_more} ({ing_more/len(recipes)*100:.1f}%)")
print(f"Inst > Ing:  {inst_more} ({inst_more/len(recipes)*100:.1f}%)")

Relation between # ingredient groups and # instruction groups:

Ing Groups   Inst Groups   # Recipes    %
--------------------------------------------------
1            1             232          15.7%  ███████████████████████████████
1            2             29           2.0%  ███
1            3             12           0.8%  █
1            4             9            0.6%  █
1            5             3            0.2%  
1            7             1            0.1%  
2            1             313          21.1%  ██████████████████████████████████████████
2            2             101          6.8%  █████████████
2            3             38           2.6%  █████
2            4             28           1.9%  ███
2            5             3            0.2%  
2            6             2            0.1%  
2            7             1            0.1%  
2            8             1            0.1%  
3            1             167          11.3%  ██████████████████████
3            2